In [1]:
import pandas as pd
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv('T8Dataset.csv')

print(df.columns)


Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')


In [3]:
df = df.drop(columns=["id","Unnamed: 32"]) 

df["diagnosis"] = df["diagnosis"].map({"M":1 , "B":0})

#splitting the features and target
X = df.drop(columns=["diagnosis"])
Y = df["diagnosis"]

X_train , X_test, Y_train, Y_test = train_test_split(X,Y,test_size = 0.2, random_state=42, stratify= Y)
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

In [4]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, Y_train)


Y_pred = knn.predict(X_test_scaled)

baseline_metrics = {
    "accuracy": accuracy_score(Y_test, Y_pred),
    "precision":precision_score(Y_test, Y_pred),
    "recall": recall_score(Y_test, Y_pred),
    "f1": f1_score(Y_test, Y_pred),
}
print(baseline_metrics)

{'accuracy': 0.956140350877193, 'precision': 0.9743589743589743, 'recall': 0.9047619047619048, 'f1': 0.9382716049382716}


In [5]:
results = []

for features in X.columns:
    X_new = X.drop(columns=[features])

    X_train , X_test, Y_train, Y_test = train_test_split(X_new,Y,test_size = 0.2, random_state=42, stratify= Y)
    scalar = StandardScaler()
    X_train_scaled = scalar.fit_transform(X_train)
    X_test_scaled = scalar.transform(X_test)

    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_scaled, Y_train)

    Y_pred = knn.predict(X_test_scaled)

    results.append({
        "removed_feature": features,
        "accuracy": accuracy_score(Y_test, Y_pred),
        "precision":precision_score(Y_test, Y_pred),
        "recall": recall_score(Y_test, Y_pred),
        "f1": f1_score(Y_test, Y_pred),
    })

results

[{'removed_feature': 'radius_mean',
  'accuracy': 0.956140350877193,
  'precision': 0.9743589743589743,
  'recall': 0.9047619047619048,
  'f1': 0.9382716049382716},
 {'removed_feature': 'texture_mean',
  'accuracy': 0.9473684210526315,
  'precision': 1.0,
  'recall': 0.8571428571428571,
  'f1': 0.9230769230769231},
 {'removed_feature': 'perimeter_mean',
  'accuracy': 0.956140350877193,
  'precision': 0.9743589743589743,
  'recall': 0.9047619047619048,
  'f1': 0.9382716049382716},
 {'removed_feature': 'area_mean',
  'accuracy': 0.956140350877193,
  'precision': 0.9743589743589743,
  'recall': 0.9047619047619048,
  'f1': 0.9382716049382716},
 {'removed_feature': 'smoothness_mean',
  'accuracy': 0.9473684210526315,
  'precision': 0.9736842105263158,
  'recall': 0.8809523809523809,
  'f1': 0.925},
 {'removed_feature': 'compactness_mean',
  'accuracy': 0.956140350877193,
  'precision': 0.9743589743589743,
  'recall': 0.9047619047619048,
  'f1': 0.9382716049382716},
 {'removed_feature': 'con

In [6]:
ablation_df = pd.DataFrame(results)
print(ablation_df)

            removed_feature  accuracy  precision    recall        f1
0               radius_mean  0.956140   0.974359  0.904762  0.938272
1              texture_mean  0.947368   1.000000  0.857143  0.923077
2            perimeter_mean  0.956140   0.974359  0.904762  0.938272
3                 area_mean  0.956140   0.974359  0.904762  0.938272
4           smoothness_mean  0.947368   0.973684  0.880952  0.925000
5          compactness_mean  0.956140   0.974359  0.904762  0.938272
6            concavity_mean  0.947368   0.973684  0.880952  0.925000
7       concave points_mean  0.956140   0.974359  0.904762  0.938272
8             symmetry_mean  0.947368   0.973684  0.880952  0.925000
9    fractal_dimension_mean  0.938596   0.972973  0.857143  0.911392
10                radius_se  0.956140   0.974359  0.904762  0.938272
11               texture_se  0.956140   0.974359  0.904762  0.938272
12             perimeter_se  0.956140   0.974359  0.904762  0.938272
13                  area_se  0.956

In [7]:
ablation_df["accuracy_drop"] = baseline_metrics["accuracy"] - ablation_df["accuracy"]
ablation_df.sort_values(by="accuracy_drop", ascending=False)


,removed_feature,accuracy,precision,recall,f1,accuracy_drop
9,fractal_dimension_mean,0.938596,0.972973,0.857143,0.911392,0.017544
26,concavity_worst,0.938596,0.972973,0.857143,0.911392,0.017544
6,concavity_mean,0.947368,0.973684,0.880952,0.925000,0.008772
8,symmetry_mean,0.947368,0.973684,0.880952,0.925000,0.008772
4,smoothness_mean,0.947368,0.973684,0.880952,0.925000,0.008772
1,texture_mean,0.947368,1.000000,0.857143,0.923077,0.008772
17,concave points_se,0.947368,0.973684,0.880952,0.925000,0.008772
19,fractal_dimension_se,0.947368,0.973684,0.880952,0.925000,0.008772
14,smoothness_se,0.947368,0.973684,0.880952,0.925000,0.008772
18,symmetry_se,0.947368,0.973684,0.880952,0.925000,0.008772
